In [ ]:
import subprocess, sys, os, site, importlib

# Install into a separate location so we don't disturb the running kernel's numpy
INSTALL_DIR = "/kaggle/working/pip-installs"
os.makedirs(INSTALL_DIR, exist_ok=True)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--target", INSTALL_DIR,
    "mediapipe==0.10.21",
])

# Prepend our install dir so our mediapipe + its numpy are found first
if INSTALL_DIR not in sys.path:
    sys.path.insert(0, INSTALL_DIR)

# Force-reload numpy so we pick up the new version from INSTALL_DIR
for mod_name in list(sys.modules.keys()):
    if mod_name == "numpy" or mod_name.startswith("numpy."):
        del sys.modules[mod_name]

import numpy as np
import mediapipe as mp
print(f"Numpy: {np.__version__}")
print(f"Mediapipe: {mp.__version__}")
print(f"Numpy loaded from: {np.__file__}")

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("MediaPipe:", mp.__version__)
print("TensorFlow:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
DATASET_ROOT = "/kaggle/input/datasets/toxicmender/20bn-jester"
FRAMES_ROOT = f"{DATASET_ROOT}/Train"
TRAIN_CSV = f"{DATASET_ROOT}/Train.csv"
VAL_CSV = f"{DATASET_ROOT}/Validation.csv"

TARGET_CLASSES = [
    "Swiping Left", "Swiping Right", 
    "Swiping Up", "Swiping Down",
    "Stop Sign", "Thumb Up", "Thumb Down",
    "No gesture", "Doing other things",
]

N_PER_CLASS = 1500
SEQ_LEN = 30
FEAT_DIM = 64

# Verify paths exist
import os
assert os.path.exists(FRAMES_ROOT), f"Missing: {FRAMES_ROOT}"
assert os.path.exists(TRAIN_CSV), f"Missing: {TRAIN_CSV}"
print(f"Target classes: {len(TARGET_CLASSES)}")
print(f"Total clips planned: {N_PER_CLASS * len(TARGET_CLASSES)}")
print("Paths OK")

In [ ]:
mp_hands = mp.solutions.hands

def extract_keypoints_from_folder(frame_folder, target_len=SEQ_LEN):
    """Reads frames from a folder, runs MediaPipe Hands on each."""
    frames = sorted(Path(frame_folder).glob("*.jpg"))
    if len(frames) == 0:
        return None
    
    idx = np.linspace(0, len(frames) - 1, target_len).astype(int)
    selected = [frames[i] for i in idx]
    
    sequence = np.zeros((target_len, FEAT_DIM), dtype=np.float32)
    
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.5,
        model_complexity=0,
    ) as hands:
        for t, frame_path in enumerate(selected):
            img = cv2.imread(str(frame_path))
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            results = hands.process(img_rgb)
            
            if results.multi_hand_landmarks:
                lm = results.multi_hand_landmarks[0].landmark
                kps = np.array([[p.x, p.y, p.z] for p in lm]).flatten()
                sequence[t, :63] = kps
                sequence[t, 63] = 1.0
    
    return sequence

print("Function defined.")

In [ ]:
# Find any folder in the frames directory
sample_folder = next(Path(FRAMES_ROOT).iterdir())
print(f"Testing on: {sample_folder}")

test_seq = extract_keypoints_from_folder(sample_folder)
print(f"Output shape: {test_seq.shape}")
print(f"Hand detected in {int(test_seq[:, 63].sum())} of {SEQ_LEN} frames")

In [ ]:
# Load with the actual CSV structure (comma-separated, has header)
train_df = pd.read_csv(TRAIN_CSV)
train_df = train_df.rename(columns={"video_id": "folder_id"})
print(f"Total training clips in dataset: {len(train_df)}")

# Filter to target classes
train_df = train_df[train_df["label"].isin(TARGET_CLASSES)]
print(f"After filtering to target classes: {len(train_df)}")

# Balance: sample N per class
balanced = (train_df.groupby("label", group_keys=False)
                    .apply(lambda x: x.sample(min(len(x), N_PER_CLASS), random_state=42))
                    .reset_index(drop=True))
print(f"Final balanced count: {len(balanced)}")
print("\nClips per class:")
print(balanced["label"].value_counts())

In [ ]:
from tqdm import tqdm
import time

# Set to None for the full run, or a small number (e.g. 100) for a test
LIMIT = None

clips_to_process = balanced if LIMIT is None else balanced.head(LIMIT)
print(f"Processing {len(clips_to_process)} clips...")

X_list, y_list = [], []
failed = 0
start = time.time()

try:
    for i, row in tqdm(clips_to_process.iterrows(), total=len(clips_to_process)):
        folder = f"{FRAMES_ROOT}/{row.folder_id}"
        try:
            seq = extract_keypoints_from_folder(folder)
            if seq is not None:
                X_list.append(seq)
                y_list.append(row.label)
            else:
                failed += 1
        except Exception as e:
            failed += 1
        
        # Checkpoint every 1000 clips
        if len(X_list) > 0 and len(X_list) % 1000 == 0:
            np.save("/kaggle/working/X_checkpoint.npy", np.stack(X_list))
            np.save("/kaggle/working/y_checkpoint.npy", np.array(y_list))

finally:
    # ALWAYS save whatever we have, even if the loop errored
    elapsed = time.time() - start
    print(f"\nDone in {elapsed/60:.1f} minutes")
    print(f"Extracted: {len(X_list)}, Failed: {failed}")
    
    if len(X_list) > 0:
        X = np.stack(X_list)
        y = np.array(y_list)
        np.save("/kaggle/working/X_keypoints.npy", X)
        np.save("/kaggle/working/y_labels.npy", y)
        print(f"Saved: X shape {X.shape}, y shape {y.shape}")
    else:
        print("Nothing to save")

In [ ]:
import numpy as np

DATA_PATH = "/kaggle/input/notebooks/eyru1317/jarvis"
X = np.load(f"{DATA_PATH}/X_keypoints.npy")
y = np.load(f"{DATA_PATH}/y_labels.npy")

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nClass distribution:")
import pandas as pd
print(pd.Series(y).value_counts())

In [ ]:
def normalize_sequence(seq):
    """Translate to wrist origin, scale by middle-finger MCP distance."""
    out = seq.copy()
    for t in range(seq.shape[0]):
        if seq[t, 63] < 0.5:  # no hand detected this frame
            continue
        coords = seq[t, :63].reshape(21, 3)
        wrist = coords[0].copy()
        coords = coords - wrist
        scale = np.linalg.norm(coords[9])
        if scale > 1e-6:
            coords = coords / scale
        out[t, :63] = coords.flatten()
    return out

X_norm = np.stack([normalize_sequence(s) for s in X])
print(f"Normalized: {X_norm.shape}")

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Label mapping:")
for i, c in enumerate(le.classes_):
    print(f"  {i}: {c}")

X_train, X_val, y_train, y_val = train_test_split(
    X_norm, y_enc, test_size=0.15, stratify=y_enc, random_state=42
)
print(f"\nTrain: {X_train.shape}")
print(f"Val:   {X_val.shape}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

SEQ_LEN, FEAT_DIM = 30, 64
NUM_CLASSES = len(le.classes_)

model = models.Sequential([
    layers.Masking(mask_value=0.0, input_shape=(SEQ_LEN, FEAT_DIM)),
    layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2),
    layers.LSTM(64, dropout=0.3, recurrent_dropout=0.2),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4, monitor="val_loss"),
    tf.keras.callbacks.ModelCheckpoint("/kaggle/working/best.keras", save_best_only=True, monitor="val_accuracy"),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = model.predict(X_val).argmax(axis=1)

print(classification_report(y_val, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", 
            xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np

OUT_DIR = "/kaggle/working"

# Save the Keras model
model.save(f"{OUT_DIR}/gesture_lstm.keras")
np.save(f"{OUT_DIR}/label_classes.npy", le.classes_)

# Export to TFLite for browser/edge deployment
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(f"{OUT_DIR}/gesture_lstm.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved artifacts:")
for f in sorted(os.listdir(OUT_DIR)):
    if not f.startswith("."):
        full = f"{OUT_DIR}/{f}"
        if os.path.isfile(full):
            size_mb = os.path.getsize(full) / 1e6
            print(f"  {f}: {size_mb:.2f} MB")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Fixes for LSTM conversion
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()
with open("/kaggle/working/gesture_lstm.tflite", "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize("/kaggle/working/gesture_lstm.tflite") / 1e6
print(f"TFLite model saved: {size_mb:.2f} MB")

In [ ]:
import os
print("Final artifacts in /kaggle/working/:")
for f in sorted(os.listdir("/kaggle/working/")):
    full = f"/kaggle/working/{f}"
    if os.path.isfile(full):
        size_mb = os.path.getsize(full) / 1e6
        print(f"  {f}: {size_mb:.2f} MB")